In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
link=f"/Volumes/aws_catalog/default/aws_databricks/project/products/*.csv"
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(link)\
    .withColumn("read_time", current_timestamp())\
    .select("*","_metadata.file_name","_metadata.file_size")
display(df)


In [0]:
df.printSchema()

In [0]:
df.write.format("delta").option('delta.enableChangeDataFeed','true').mode("overwrite")\
    .saveAsTable("fmcg.bronze.products")

In [0]:
df_bronze=spark.read.table("fmcg.bronze.products")
df_silver=df_bronze.dropDuplicates(["product_id"])
df_silver.count()

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver=df_silver.withColumn("category",initcap(col("category")))
df_silver.select("category").distinct().show()

In [0]:

df_silver=df_silver.withColumn("category",regexp_replace(col("category"),"Protien","Protein"))\
    .withColumn("product_name",regexp_replace(col("product_name"),"rotien","Organic Food"))
display(df_silver)

In [0]:
df_silver=(df_silver.withColumn("division",
                     when(col("category") == "Energy Bars",        "Nutrition Bars")
                    .when(col("category") == "Protein Bars",       "Nutrition Bars")
                    .when(col("category") == "Granola & Cereals",  "Breakfast Foods")
                    .when(col("category") == "Recovery Dairy",     "Dairy & Recovery")
                    .when(col("category") == "Healthy Snacks",     "Healthy Snacks")
                    .when(col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
                    .otherwise("Other")
                     )
)
display(df_silver)

In [0]:
df_silver=df_silver.withColumn("variant", regexp_extract(col("product_name"), r"\((.*?)\)", 1))
df_silver=df_silver.withColumn("product_code", sha2(col("product_name").cast("string"),256))
display(df_silver.limit(5))

In [0]:
df_silver=df_silver.withColumn(
        "product_id",
        when(col("product_id").cast("string").rlike("^[0-9]+$"), col("product_id").cast("string")
        ).otherwise(lit(999999).cast("string"))
)
display(df_silver)

In [0]:
df_silver.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite").saveAsTable("fmcg.silver.products")

In [0]:
df_silver=spark.read.table("fmcg.silver.products")
df_gold=df_silver.select("product_code", "product_id", "division", "category", "product_name", "variant")
df_gold=df_gold.withColumnRenamed("product_name", "product")
display(df_gold)

In [0]:
df_gold.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite").saveAsTable("fmcg.gold.sb_dim_products")

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_products")
child_table=spark.read.table("fmcg.gold.sb_dim_products")\
    .select("product_code","division","category","product","variant")

delta_table.alias("target").merge(child_table.alias("source"), "target.product_code=source.product_code")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()